## Target exploration

Track A predicts seven simulator parameters:
planetary radius, atmospheric
temperature and five molecular log-abundances. Their order is defined
explicitly because every later target array, model output and inverse
transformation must use the same order.

This section verifies that all required targets exist and contain finite numeric
values. It then summarises their distributions and numerical scales. These are
source-data EDA statistics only(!), they will not be used as fitted preprocessing
parameters.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TARGET_PATH = (
    PROJECT_ROOT
    / "FullDataset"
    / "TrainingData"
    / "Ground Truth Package"
    / "FM_Parameter_Table.csv"
)

if not TARGET_PATH.is_file():
    raise FileNotFoundError(
        f"Simulator target table not found: {TARGET_PATH}"
    )

targets = pd.read_csv(TARGET_PATH)

print("Target file:", TARGET_PATH)
print("Raw target-table shape:", targets.shape)
print("Raw columns:", targets.columns.tolist())

Target file: /home/ajf23/Documents/Coding new/University stuff/ADC2023/ADC2023-baseline/FullDataset/TrainingData/Ground Truth Package/FM_Parameter_Table.csv
Raw target-table shape: (41423, 9)
Raw columns: ['Unnamed: 0', 'planet_ID', 'planet_radius', 'planet_temp', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']


## Define and validate targets

In [3]:
# Fixes target order
TARGET_COLUMNS = [
    "planet_radius",
    "planet_temp",
    "log_H2O",
    "log_CO2",
    "log_CO",
    "log_CH4",
    "log_NH3",
]

missing_target_columns = [column for column in TARGET_COLUMNS
                          if column not in targets.columns]

if missing_target_columns:
    raise ValueError(
        "The simulator target table is missing required columns:" f"{missing_target_columns}"
    )

# Creates a target-only DF shape (41423, 7)
target_only = targets.loc[:, TARGET_COLUMNS].copy()

# Verify every selected value is numeric
try:
    target_only = target_only.apply(pd.to_numeric, errors="raise")

except (TypeError, ValueError) as error:
    raise ValueError("One or more target columns have non-numeric values.") from error

target_values = target_only.to_numpy(dtype=np.float64)
finite_mask = np.isfinite(target_values)

if not finite_mask.all():
    non_finite_counts = {column: int((~finite_mask[:, index]).sum())
                         for index, column in enumerate(TARGET_COLUMNS)
                         if (~finite_mask[:, index]).any()
                         }
    raise ValueError(
        "Non finite target values were detected: "
        f"{non_finite_counts}"
    )

print("Target DataFrame shape:", target_only.shape)
print("Target columns:", target_only.columns.tolist())
print("All target values are numeric and finite.")

Target DataFrame shape: (41423, 7)
Target columns: ['planet_radius', 'planet_temp', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
All target values are numeric and finite.


## Summary stats and ranges

The targets have numerical scales.

For example, atmospheric temperature is measured on a much larger numerical
scale than planetary radius or a logarithmic abundance. If these targets were
used directly in a multi-output loss, the larger-scale targets could dominate.

The data pipeline will therefore standardise each target separately.
However, its means and standard deviations must be fitted using the future
training data only. Using statistics from validation, calibration or test
targets would leak information from those into model training.

In [ ]:
summary_columns = [
    "count",
    "mean",
    "std",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
]


# Transposing (.T) places one target on each row, which makes comparisons easier.
target_summary = (target_only.describe(percentiles=[0.25, 0.50, 0.75]).T.loc[:, summary_columns]
                  .rename(columns={
                      "min": "minimum",
                      "25%": "25th_percentile",
                      "50%": "median",
                      "75%": "75th_percentile",
                      "max": "maximum"
                  }))

target_ranges = pd.DataFrame({"minimum": target_only.min(),
                              "maximum": target_only.max()})

target_ranges["range"] = (target_ranges["maximum"] -target_ranges["minimum"])

print("Target summary:")
display(target_summary)

print("\nTarget ranges:")
display(target_ranges)

Target summary:


,count,mean,std,minimum,25th_percentile,median,75th_percentile,maximum
planet_radius,41423.0,0.655128,0.453444,0.075000,0.247160,0.491891,1.037999,2.429861
planet_temp,41423.0,1004.971922,401.278224,101.021245,699.502902,912.143137,1213.682913,4965.288165
log_H2O,41423.0,-5.982997,1.733389,-8.999856,-7.482630,-5.968308,-4.482412,-3.000497
log_CO2,41423.0,-6.522149,1.437302,-8.999932,-7.766406,-6.524401,-5.291455,-4.000009
log_CO,41423.0,-4.491494,0.869522,-5.999984,-5.241152,-4.486111,-3.734558,-3.000010
log_CH4,41423.0,-5.997146,1.731234,-8.999856,-7.489864,-6.005781,-4.494879,-3.000333
log_NH3,41423.0,-6.494065,1.456278,-8.999985,-7.767215,-6.489616,-5.249536,-3.004636



Target ranges:


,minimum,maximum,range
planet_radius,0.075000,2.429861,2.354861
planet_temp,101.021245,4965.288165,4864.266920
log_H2O,-8.999856,-3.000497,5.999358
log_CO2,-8.999932,-4.000009,4.999923
log_CO,-5.999984,-3.000010,2.999973
log_CH4,-8.999856,-3.000333,5.999524
log_NH3,-8.999985,-3.004636,5.995349
